<a href="https://colab.research.google.com/github/pythonWolf59/PythonPortfolio/blob/Machine-Learning/Financial_Markets_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# START HERE

**Run this cell to install necessary libraries**

In [13]:
#install libraries
!pip install ta datasets --quiet
!pip install pandas_datareader --quiet
!pip install xgboost scikit-learn joblib --quiet


# DATA PRE PROCESSING

In [ ]:
import pandas as pd
import numpy as np

# Load stock data
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Datasets/combined_stock_data.csv", parse_dates=['Date'])
# stocks_etf = cudf.read_csv("/content/drive/MyDrive/Colab Notebooks/Datasets/stock_etf_data.csv")

# Identify the correct 'Close' column - assuming the first one is the intended one
close_column = [col for col in df.columns if 'Close' in col][0] # Gets the first column name containing 'Close'

# Convert the identified 'Close' column to numeric, coercing errors
df[close_column] = pd.to_numeric(df[close_column], errors='coerce')

# Rename the selected close column to just 'Close' for consistency
df = df.rename(columns={close_column: 'Close'})

# Select only the relevant columns to avoid interference from other data types
df = df[['Date', 'Ticker', 'Close']].copy()

# Drop rows where 'Close' is NaN *before* sorting and grouping
df = df.dropna(subset=['Close'])

# Sort for accurate calculations within groups
df = df.sort_values(['Ticker', 'Date'])

# Calculate daily log returns as the difference of log prices
# Calculate log of 'Close' price first
df['LogClose'] = df.groupby('Ticker')['Close'].transform(lambda x: np.log(x))

# Calculate log return as the difference of shifted log prices
df['LogReturn'] = df.groupby('Ticker')['LogClose'].transform(lambda x: x - x.shift(1))

# Drop the intermediate 'LogClose' column
df = df.drop(columns=['LogClose'])


# Define volatility target: rolling std of returns using transform
window_size = 10  # You can tune this (5, 10, 20, etc.)
df['VolatilityTarget'] = df.groupby('Ticker')['LogReturn'].transform(lambda x: x.rolling(window_size).std())

# Optional: shift the target to predict future volatility
df['VolatilityTarget'] = df.groupby('Ticker')['VolatilityTarget'].shift(-window_size)

# Drop rows with NaNs (due to shifting and rolling) - this should primarily remove NaNs from the rolling window
df = df.dropna(subset=['LogReturn', 'VolatilityTarget'])


# Save for next steps
df.to_csv("/content/drive/MyDrive/Colab Notebooks/Datasets/volatility_target.csv", index=False)

print(df.head())

In [ ]:
import pandas as pd
import ta

# === STEP 0: Load main dataset ===
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Datasets/volatility_target.csv", parse_dates=["Date"])
df = df.sort_values(['Ticker', 'Date'])

# === STEP 1: Add Technical Indicators ===
def add_technical_indicators(group):
    group = group.copy()
    group['RSI'] = ta.momentum.RSIIndicator(close=group['Close'], window=14).rsi()
    macd = ta.trend.MACD(close=group['Close'])
    group['MACD'] = macd.macd()
    bb = ta.volatility.BollingerBands(close=group['Close'])
    group['BB_High'] = bb.bollinger_hband()
    group['BB_Low'] = bb.bollinger_lband()
    group['EMA_20'] = group['Close'].ewm(span=20).mean()
    group['EMA_50'] = group['Close'].ewm(span=50).mean()
    return group

df = df.groupby('Ticker', group_keys=False).apply(add_technical_indicators)
df = df.dropna()  # Drop rows with NA values from indicators

# === STEP 3: Merge Macroeconomic Indicators ===
macro_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Datasets/macro_indicators.csv", parse_dates=["DATE"])
macro_df = macro_df.rename(columns={"DATE": "Date"})

# Resample to daily and forward-fill values
macro_df = macro_df.set_index("Date").resample("D").ffill().reset_index()

# Merge with main dataset
df = df.merge(macro_df, on="Date", how="left")
df = df.fillna(method='ffill')  # Final cleanup

# Fill GDP gaps
df['GDP'] = df['GDP'].fillna(method='ffill')

# === Save Final Dataset ===
df.to_csv("/content/drive/MyDrive/Colab Notebooks/Datasetsfinal_model_data.csv", index=False)

# Quick preview
print(df.info())

# BASE MODEL

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import numpy as np

# Load cleaned dataset
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Datasets/final_model_data.csv", parse_dates=["Date"])

# Drop columns not used for training
X = df.drop(columns=['Date', 'VolatilityTarget', 'Ticker'])  # drop non-numeric
y = df['VolatilityTarget']

#Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Model

model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# Model Evaluation
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.6f}")
print(f"R² Score: {r2:.4f}")

# Save Model

joblib.dump(model, "/content/drive/MyDrive/Colab Notebooks/Datasets/volatility_gbm_model.pkl")
print("Model saved as volatility_gbm_model.pkl")


# FEATURE ENGINEERING

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
import joblib

# Load and sort
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Datasets/final_model_data.csv", parse_dates=["Date"])
df = df.sort_values("Date")

# Existing lags and rolling std
for f in ['LogReturn', 'RSI', 'MACD']:
    df[f'{f}_lag1'] = df[f].shift(1)
    df[f'{f}_lag2'] = df[f].shift(2)
    df[f'{f}_roll3std'] = df[f].rolling(3).std()

# New: 3-day % trend of Close
df['Close_trend3'] = df['Close'].pct_change(periods=3)

# New: 5-day EMA of Close
df['Close_ema5'] = df['Close'].ewm(span=5, adjust=False).mean()

# New: Lag of VolatilityTarget (past volatility)
df['VolatilityTarget_lag1'] = df['VolatilityTarget'].shift(1)

# Date-based feature
df['day_of_week'] = df['Date'].dt.dayofweek

# Define feature columns
features = [
    'Close', 'LogReturn', 'RSI', 'MACD',
    'CPI', 'Unemployment', 'GDP', 'Interest Rate',
    'LogReturn_lag1', 'RSI_lag1', 'MACD_lag1',
    'LogReturn_lag2', 'RSI_lag2', 'MACD_lag2',
    'LogReturn_roll3std', 'RSI_roll3std', 'MACD_roll3std',
    'day_of_week', 'Close_trend3', 'Close_ema5',
    'VolatilityTarget_lag1'
]

# Drop rows with NaNs from lagging and rolling
df = df.dropna(subset=features + ['VolatilityTarget'])

# Target
target = 'VolatilityTarget'

# Train/test split
split_idx = int(len(df) * 0.85)
X_train = df.iloc[:split_idx][features]
y_train = df.iloc[:split_idx][target]
X_test  = df.iloc[split_idx:][features]
y_test  = df.iloc[split_idx:][target]

# LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_test, label=y_test)

# Params
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.03,
    'num_leaves': 64,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

# Train
model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    num_boost_round=1000,
)

# Predict & Evaluate
y_pred = model.predict(X_test, num_iteration=model.best_iteration)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"✅ RMSE on test set: {rmse:.6f}")
print(f"✅ R² on test set: {r2:.4f}")

# --- Save Model ---
joblib.dump(model, "/content/drive/MyDrive/Colab Notebooks/Datasets/volatility_gbm_finetuned.pkl")
print("✅ Model saved as volatility_gbm_finetuned.pkl")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_test.values[:250], label='Actual', marker='o')
plt.plot(y_pred[:250], label='Predicted', marker='x')
plt.legend()
plt.title("Actual vs Predicted Volatility (First 250 Samples)")
plt.show()


# PREDICT ON NEW DATA

https://fredaccount.stlouisfed.org/
Sign in with Google

In [ ]:
# Fix numpy/pandas binary incompatibility by reinstalling and restarting runtime
!pip install --upgrade --force-reinstall numpy pandas pandas_ta lightgbm ipywidgets alpha_vantage requests --quiet

# Restart Runtime after running this cell


In [ ]:
import pandas as pd
import numpy as np
import pandas_ta as ta
from alpha_vantage.timeseries import TimeSeries
import lightgbm as lgb
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests

# === API KEYS - Replace these with your own keys ===
ALPHA_VANTAGE_API_KEY = "V7EJYS5SS0YR46WT"
FRED_API_KEY = "1a1f0ece79980e29bfbe7e209ae49b8d"

# Initialize Alpha Vantage
ts = TimeSeries(key=ALPHA_VANTAGE_API_KEY, output_format='pandas')

def fetch_alpha_vantage_daily(ticker):
    try:
        data, _ = ts.get_daily_adjusted(symbol=ticker, outputsize='compact')
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        return pd.DataFrame()
    data = data.rename(columns={
        '5. adjusted close': 'Close',
        '1. open': 'Open',
        '2. high': 'High',
        '3. low': 'Low',
        '6. volume': 'Volume'
    })
    data.index = pd.to_datetime(data.index)
    data = data.sort_index()
    return data[['Close', 'Open', 'High', 'Low', 'Volume']]

def add_features(df):
    df['LogReturn'] = np.log(df['Close']).diff()
    df['RSI'] = ta.rsi(df['Close'], length=14)
    macd_df = ta.macd(df['Close'])
    df['MACD'] = macd_df['MACD_12_26_9'] if macd_df is not None else np.nan
    for f in ['LogReturn', 'RSI', 'MACD']:
        df[f'{f}_lag1'] = df[f].shift(1)
        df[f'{f}_lag2'] = df[f].shift(2)
        df[f'{f}_roll3std'] = df[f].rolling(3).std()
    df['Close_trend3'] = df['Close'].pct_change(3)
    df['Close_ema5'] = df['Close'].ewm(span=5, adjust=False).mean()
    df['day_of_week'] = df.index.dayofweek
    df.dropna(inplace=True)
    return df

def fetch_macro_data():
    def get_fred_series(series_id):
        url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json"
        r = requests.get(url)
        data = r.json()['observations']
        df = pd.DataFrame(data)
        df['date'] = pd.to_datetime(df['date'])
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df = df.dropna(subset=['value'])
        return df.set_index('date')['value']

    macro = {}
    macro['CPI'] = get_fred_series('CPIAUCSL').last('1M').mean()
    macro['Unemployment'] = get_fred_series('UNRATE').last('1M').mean()
    macro['Interest Rate'] = get_fred_series('FEDFUNDS').last('1M').mean()
    macro['GDP'] = get_fred_series('GDP').resample('Q').mean().last('1Q').mean()
    return macro

# Load your LightGBM model using joblib
model = joblib.load("/content/drive/MyDrive/Colab Notebooks/Datasets/volatility_gbm_finetuned.pkl")

feature_cols = [
    'Close', 'LogReturn', 'RSI', 'MACD',
    'CPI', 'Unemployment', 'GDP', 'Interest Rate',
    'LogReturn_lag1', 'RSI_lag1', 'MACD_lag1',
    'LogReturn_lag2', 'RSI_lag2', 'MACD_lag2',
    'LogReturn_roll3std', 'RSI_roll3std', 'MACD_roll3std',
    'day_of_week', 'Close_trend3', 'Close_ema5'
]

ticker_input = widgets.Text(
    value='AAPL',
    description='Ticker:',
    placeholder='Enter ticker symbol',
    layout=widgets.Layout(width='50%')
)
horizon_dropdown = widgets.Dropdown(
    options=[1, 2, 3, 5, 10],
    value=3,
    description='Days:'
)
button = widgets.Button(description="Predict Volatility")
output = widgets.Output()

macro_cache = None  # cache macro data for session

def on_button_click(b):
    global macro_cache
    with output:
        clear_output()
        ticker = ticker_input.value.strip().upper()
        if not ticker:
            print("❌ Please enter a ticker symbol.")
            return

        print(f"Fetching data for {ticker} ...")
        df = fetch_alpha_vantage_daily(ticker)
        if df.empty:
            print(f"❌ No data available for ticker {ticker}.")
            return

        print("Adding features...")
        df = add_features(df)
        if df.empty:
            print("❌ Not enough data after feature engineering to predict.")
            return

        latest = df.iloc[-1:].copy()

        if macro_cache is None:
            print("Fetching macroeconomic data from FRED...")
            macro_cache = fetch_macro_data()

        for k, v in macro_cache.items():
            latest[k] = v

        missing = set(feature_cols) - set(latest.columns)
        if missing:
            print(f"❌ Missing features for prediction: {missing}")
            return

        X_latest = latest[feature_cols]

        pred_vol = model.predict(X_latest)[0]
        print(f"📈 Predicted {horizon_dropdown.value}-day volatility for {ticker}: {pred_vol * 100:.4f}%")

button.on_click(on_button_click)
display(ticker_input, horizon_dropdown, button, output)